In [56]:
import pandas as pd

In [57]:
audi = pd.read_csv("../data/audi.csv")
bmw = pd.read_csv("../data/bmw.csv")
ford = pd.read_csv("../data/ford.csv")
hyundai = pd.read_csv("../data/hyundi.csv")
mercedes = pd.read_csv("../data/merc.csv")
skoda = pd.read_csv("../data/skoda.csv")
toyota = pd.read_csv("../data/toyota.csv")
vauxhall = pd.read_csv("../data/vauxhall.csv")
volkswagen = pd.read_csv("../data/vw.csv")

In [58]:
audi["make"] = "Audi"
bmw["make"] = "BMW"
ford["make"] = "Ford"
hyundai["make"] = "Hyundai"
mercedes["make"] = "Mercedes"
skoda["make"] = "Skoda"
toyota["make"] = "Toyota"
vauxhall["make"] = "Vauxhall"
volkswagen["make"] = "Volkswagen"

In [59]:
#before making combined dataset, had to rename column in hyundai df as it has column names tax(£) instead of just tax
hyundai = hyundai.rename(columns={'tax(£)': 'tax'})

In [60]:
#combining all the cars data together
cars = pd.concat(
    [audi, bmw, ford, hyundai, mercedes,
     skoda, toyota, vauxhall, volkswagen],
    ignore_index=True
)

In [61]:
#based on observations, these do look like duplications so will drop duplicate rows
cars = cars.drop_duplicates()

Duplicate Rows
The dataset contained 1,475 exact duplicate rows.

As there is no unique vehicle or listing identifier, it is not possible to determine whether these represent genuinely different vehicles with identical characteristics or duplicated observations.

The duplicate rows were removed to:

Prevent repeated observations from disproportionately influencing model training.
Reduce the risk of identical observations appearing in both the training and test sets.

In [63]:
cars = cars.copy()
cars["model"] = cars["model"].str.lstrip()

In [64]:
cars = cars[cars["transmission"]!="Other"] 
#dropping rows where the transmission type is other (only 9 entries)

In [65]:
cars = cars[cars["fuelType"]!="Other"] 
#where fuel type is other this is spread over a range of different cars, and since 245 is a very small proportion of the entire dataset (0.25%)
#decision made to drop rows where fuel type is other

In [66]:
cars = cars[cars["fuelType"]!="Electric"] 
#E lectric vehicles were removed because the category contained only six observations (<0.01% of the dataset)
# and inspection indicated inconsistent fuel-type classification.

In [67]:
cars

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,make
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,Audi
...,...,...,...,...,...,...,...,...,...,...
99182,Eos,2012,5990,Manual,74000,Diesel,125,58.9,2.0,Volkswagen
99183,Fox,2008,1799,Manual,88102,Petrol,145,46.3,1.2,Volkswagen
99184,Fox,2009,1590,Manual,70000,Petrol,200,42.0,1.4,Volkswagen
99185,Fox,2006,1250,Manual,82704,Petrol,150,46.3,1.2,Volkswagen


In [68]:
cars = cars[cars["year"].between(1996, 2020)] 
#only selecting date range between 1996-2020 as exploration of dates shows that there are some erroneous dates
#e.g. cars with year 1970 where investigation shows that model of car had only been in production in the 1990s. 2060 is listed as a year

In [69]:
cars = cars[cars["engineSize"]!=0]
#Removed 264 records (0.27%) with an engine size of 0, as these represented implausible or missing values and comprised a negligible proportion 
#of the dataset.

In [70]:
#early cars with mileage = 1 look implausible when considering their price, removed these three observations
cars = cars[~((cars["mileage"] == 1) & (cars["year"] <= 2008))]

In [71]:
cars = cars[cars["mpg"] >= 10]
#mpgs of less than 10 mpg look erroneous. decided to remove as very small percentage of total 33/99000

In [72]:
cars = cars[
    ~(
        (cars["mpg"] > 150) &
        (cars["fuelType"].isin(["Petrol", "Diesel"]))
    )
]
#removed abnormally high mpg from dataset where fuel type is Petrol/Diesel (3 observations), as these appear erroneous

In [73]:
cars = cars.reset_index(drop=True)

In [74]:
cars

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,make
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,Audi
...,...,...,...,...,...,...,...,...,...,...
97141,Eos,2012,5990,Manual,74000,Diesel,125,58.9,2.0,Volkswagen
97142,Fox,2008,1799,Manual,88102,Petrol,145,46.3,1.2,Volkswagen
97143,Fox,2009,1590,Manual,70000,Petrol,200,42.0,1.4,Volkswagen
97144,Fox,2006,1250,Manual,82704,Petrol,150,46.3,1.2,Volkswagen


In [75]:
cars.to_csv("../data/used_cars_cleaned.csv")